# EP 1 - Il tuo primo agente con Microsoft Agent Framework

Costruiamo un agente in poche righe, lo eseguiamo su OpenAI e poi - con lo stesso codice - su Ollama in locale. Niente tool: quelli arrivano in EP 2.

Companion teorico: `teoria.md`. Notebook pensato per **Colab e VS Code**: la cella di setup rileva l'ambiente.

## 1. Setup

Rileviamo l'ambiente, installiamo le dipendenze dove serve e carichiamo la chiave OpenAI.

> In VS Code si presuppone l'ambiente `uv` (`uv add agent-framework-core agent-framework-openai agent-framework-ollama python-dotenv`). Su Colab la cella installa da sola il minimo.
>
> Promemoria: **MAF non carica `.env` da solo**, serve `load_dotenv()`.

In [16]:
import sys, subprocess, os, warnings

# MAF segnala come "experimental" feature che qui non usiamo: silenziamo il rumore in output.
warnings.filterwarnings("ignore", message=".*experimental.*")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Su Colab installiamo il minimo per il percorso OpenAI (Ollama gira solo in locale).
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "agent-framework-core", "agent-framework-openai", "python-dotenv"],
        check=True,
    )

from dotenv import load_dotenv
load_dotenv()  # MAF non carica .env da solo

# Chiave OpenAI: da .env in locale, chiesta a mano su Colab
if not os.environ.get("OPENAI_API_KEY"):
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

MODEL = "gpt-4o-mini"  # cambia col modello che preferisci
print("Ambiente:", "Colab" if IN_COLAB else "locale (VS Code/uv)")

Ambiente: locale (VS Code/uv)


## 2. Il primo agente

Un agente è un LLM con un ruolo, eseguito da un runtime che gestirà il loop quando servirà (da EP 2, con i tool). Senza tool il loop fa un solo giro: è il momento giusto per capire l'oggetto `Agent`.

Tre parti: **chat client** (il provider) -> **Agent** (identità + istruzioni) -> **run** (l'esecuzione). Costruiamo il client **una volta** e lo riusiamo per tutti gli agenti. Le `instructions` sono la leva principale: cambi quelle e cambi l'agente.

`run` è asincrono. In notebook l'event loop è già attivo, quindi usiamo direttamente `await agent.run(...)`.

In [17]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

client = OpenAIChatClient(model=MODEL)             # il provider: costruito una volta, riusato ovunque

agent = Agent(
    client=client,                                 # 1. il provider
    name="Opinionista",                            # 2a. identità
    instructions=(                                 # 2b. comportamento
        "Sei un critico cinematografico tagliente. "
        "Difendi la tua tesi con argomenti concreti, in italiano, senza giri di parole."
    ),
)

risposta = await agent.run("In una frase: qual è il film più sopravvalutato di sempre?")  # 3. esecuzione
print(risposta.text)

"Via col vento" è il film più sopravvalutato di sempre, poiché glorifica una visione romantica e distorta della schiavitù, mascherando una narrazione superficiale dietro una cinematografia affascinante.


## 3. Streaming

In streaming la risposta appare man mano che viene generata: esperienza migliore con risposte lunghe. Stesso agente, cambia solo come leggiamo l'output: `stream=True` e iteriamo gli update. (Non esiste `run_stream`: è sempre `run`.)

> Nota importante: questo è un `run` **indipendente** dal precedente. In EP 1 non passiamo una sessione, quindi l'agente **non ricorda** cosa ha detto prima: ogni chiamata riparte da zero. Per questo il prompt qui è autoconsistente (nomina il film). La memoria tra i turni arriva in EP 3.

In [20]:
print(f"{agent.name}: ", end="")
async for update in agent.run("Argomenta in tre punti perché 'The Happening' è sopravvalutato.", stream=True):
    if update.text:
        print(update.text, end="", flush=True)
print()

Opinionista: 1. **Sceneggiatura debole**: La trama di 'The Happening' è poco coerente e piena di buchi logici. La premessa di un misterioso agente atmosferico che provoca suicidi di massa è affascinante, ma la sua esecuzione è semplicistica e priva di approfondimento, lasciando lo spettatore confuso e insoddisfatto.

2. **Recitazione carente**: Le performance degli attori, in particolare quella di Mark Wahlberg, sono poco convincenti e spesso ridicole. Le reazioni dei personaggi alle situazioni drammatiche risultano forzate e poco realistiche, contribuendo a una sensazione di disconnessione emotiva con il pubblico.

3. **Scelte stilistiche poco convincenti**: M. Night Shyamalan tenta di costruire suspense con tecniche visive che invece finiscono per risultare scontate e inefficaci. La mancanza di tensione reale e di atmosfere coinvolgenti rende 'The Happening' più un esercizio di stile fallito che un vero thriller, impoverendo l'esperienza cinematografica complessiva.


## 4. Stesso agente, modello in locale (Ollama)

L'agente non sa quale modello ha sotto: cambiamo il chat client, il resto resta identico. È il "no lock-in" reso concreto: cloud quando conta la qualità, locale quando contano costo zero e privacy.

Due strade per puntare a Ollama, entrambe vere:
- **A) client nativo** `OllamaChatClient`: il più diretto, nessuna chiave.
- **B) lo stesso** `OpenAIChatClient` **puntato all'endpoint locale**: cambia solo l'endpoint. È la dimostrazione più forte del no lock-in, e quella che eseguiamo qui.

Bonus per VS Code (su Colab si salta). Prerequisiti: [Ollama](https://ollama.com) avviato (`ollama serve` da terminale) e `ollama pull llama3.2` (`ollama list` mostra i modelli che hai). Scegli un modello che entra nella tua GPU: un 3B come llama3.2 è veloce; modelli grossi o "reasoning" rallentano molto.

In [22]:
if IN_COLAB:
    print("Ollama gira in locale: salta su Colab, prova questa cella in VS Code.")
else:
    # A) client nativo Ollama: il più diretto, nessuna chiave.
    from agent_framework.ollama import OllamaChatClient
    client_ollama_a = OllamaChatClient(model="llama3.2")

    # B) lo STESSO OpenAIChatClient di prima, puntato all'endpoint locale: cambia solo l'endpoint.
    client_ollama_b = OpenAIChatClient(
        base_url="http://localhost:11434/v1/",
        model="llama3.2",
        api_key="ollama",      # Ollama non usa la chiave, ma il client la vuole valorizzata
    )

    # Eseguiamo la strada B (dimostrazione più forte). Per provare la A: client=client_ollama_a.
    ollama_agent = Agent(
        client=client_ollama_b,
        name="Opinionista",
        instructions="Sei un critico cinematografico tagliente. Rispondi in italiano, conciso.",
    )
    try:
        risposta_locale = await ollama_agent.run("In una frase: qual è il film più sopravvalutato di sempre?")
        print(risposta_locale.text)
    except Exception as e:
        print("Ollama non raggiungibile. Avvialo da terminale con `ollama serve`,")
        print("e assicurati di aver scaricato il modello con `ollama pull llama3.2`.")
        print("Dettaglio:", e)

Secondo me, il film più sopravvalutato è "Il Signore degli Anelli" del 2001 diretto da Peter Jackson, che ha ricevuto innumerevoli premi e accontiati come uno dei migliori film di tutti i tempi.


## 5. Verso il Debate Club: due personalità, una motion

La personalità di un agente vive nelle `instructions`. Stesso modello, stesso codice: cambi le istruzioni e ottieni agenti opposti sulla stessa tesi. È l'anteprima del Debate Club (EP 4). Prova poi a cambiare istruzioni o motion.

In [23]:
fan = Agent(
    client=client,                                 # stesso client di prima
    name="Fan",
    instructions=(
        "Sei un fan accanito di M. Night Shyamalan. Difendi anche i suoi film più criticati, "
        "trovando sempre il lato geniale. In italiano, appassionato ma concreto."
    ),
)

critico = Agent(
    client=client,                                 # stesso client, agente opposto
    name="Critico",
    instructions=(
        "Sei un critico severo di M. Night Shyamalan. Argomenta perché i suoi film più discussi "
        "non funzionano, con esempi precisi. In italiano, tagliente ma onesto."
    ),
)

motion = "The Happening è il peggior film di Shyamalan."

print("== Fan ==")
risposta_fan = await fan.run(motion)
print(risposta_fan.text)
print("\n== Critico ==")
risposta_critico = await critico.run(motion)
print(risposta_critico.text)

== Fan ==
Capisco le critiche che ha ricevuto "The Happening", ma lasciami difendere questo film con passione! Prima di tutto, Shyamalan non ha mai temuto di affrontare tematiche complesse, e in "The Happening" ci invita a riflettere su problemi ambientali e sulle conseguenze delle azioni umane. 

La scelta di un attacco "naturale" attraverso il rilascio di una tossina è una metafora potente: rappresenta la reazione della natura a un'umanità distruttiva. Non è un horror convenzionale, ma un thriller psicologico che esplora la paura e l'ignoto.

Inoltre, le performance di Mark Wahlberg e Zooey Deschanel possono sembrare bizzarre, ma sono parte di una narrazione che gioca con l’assurdo. Shyamalan ci fa sentire a disagio, proprio come i personaggi stessi. La direzione artistica e la fotografia creano un'atmosfera inquietante, amplificando il senso di impotenza di fronte a una crisi. 

Infine, il film è pregno di momenti che stimolano la discussione: cosa succederebbe se davvero la natura 

## 6. Far uscire l'output: una notifica push

Finora la risposta resta nel notebook. Diamole una via d'uscita: una notifica push sul telefono con [Pushover](https://pushover.net).

Una sfumatura che conta: **questo non è un tool**. Qui è il *nostro codice* a mandare la notifica dopo che l'agente ha risposto, sempre. Da EP 2, con i tool, sarà l'**agente** a decidere se e quando notificare. Stesso Pushover, ruolo diverso.

Serve un account Pushover (gratis): crea un'app per avere il `token`, copia il tuo `user` key, e mettili nel `.env` come `PUSHOVER_TOKEN` e `PUSHOVER_USER`. Senza, la cella te lo dice e salta.

In [24]:
import os, httpx

def notifica(messaggio: str, titolo: str = "Fan") -> None:
    """Manda una push via Pushover, se le chiavi sono nel .env.
    È il NOSTRO codice a mandarla: non è (ancora) un tool dell'agente."""
    token, user = os.environ.get("PUSHOVER_TOKEN"), os.environ.get("PUSHOVER_USER")
    if not (token and user):
        print("Pushover non configurato: aggiungi PUSHOVER_TOKEN e PUSHOVER_USER al .env")
        return
    r = httpx.post(
        "https://api.pushover.net/1/messages.json",
        data={"token": token, "user": user, "message": messaggio[:1024], "title": titolo},
    )
    print("Notifica inviata!" if r.json().get("status") == 1 else f"Errore Pushover: {r.text}")

# mandiamo come push la risposta del Fan (dalla sez. 5)
# prova a cambiare risposta_fan -> risposta_critico: stesso codice, altro agente
notifica(risposta_fan.text, titolo=fan.name)

Notifica inviata!


## 7. Dal notebook al terminale: un programma vero

Il notebook è perfetto per esplorare, ma un agente è codice che gira. Lo stesso agente vive anche come script standalone, **`agente.py`**, in questa cartella. La differenza è solo l'involucro: in notebook l'event loop è già attivo (`await` diretto), in uno script lo avvii tu con `asyncio.run(main())`.

Aprilo, poi da un terminale vero lancia:

```bash
uv run python agente.py
```

Lo script fa un passo in più del notebook: fa rispondere **entrambi** gli agenti (Fan e Critico) sulla stessa motion, **salva ogni risposta** in `output_<nome>.md` e (se Pushover è configurato) **manda una push per ciascuno**. È il nostro primo programma agentico completo, fuori dal notebook.

## Fatto

Hai un agente che gira su cloud e in locale con lo stesso codice. In **EP 2** gli diamo gli strumenti (tool + MCP) e il loop agentico si accende davvero.